## Generación de Señales 

## Sag


In [ ]:
import numpy as np
import pandas as pd

def sag(amplitud, frecuencia, fase, tiempo, alpha, t1, t2):
    """Genera una señal de hundimiento de tensión."""
    titulo = "Hundimiento de Tensión"
    wt = 2 * np.pi * frecuencia * tiempo
    u1 = np.heaviside(tiempo - t1, 1)  # u(t - t1)
    u2 = np.heaviside(tiempo - t2, 1)  # u(t - t2)
    envolvente = 1 - (alpha * (u1 - u2))
    señal = amplitud * envolvente * np.sin(wt + fase)
    return señal, titulo

# --- Parámetros de Generación ---
num_sags = 1100
num_puntos = 7200
duracion_total = 1.0 # en segundos
fs = num_puntos / duracion_total # Frecuencia de muestreo

tiempo = np.linspace(0, duracion_total, num_puntos)
sags_df = pd.DataFrame()


metadata_list = []

for i in range(num_sags):
    # Genera parámetros aleatorios para cada sag
    amplitud = 1.0
    frecuencia = 60.0
    fase = np.random.uniform(0, 2 * np.pi)
    alpha = np.random.uniform(0.1, 0.9)
    t1 = np.random.uniform(0.1, 0.4)
    t2 = t1 + np.random.uniform(0.1, 0.4)

    # Genera la señal del sag
    nombre_sag = f'Sag_{i+1}'
    sag_signal, _ = sag(amplitud, frecuencia, fase, tiempo, alpha, t1, t2)
    sags_df[nombre_sag] = sag_signal
    
    # Calcula los índices de inicio y fin en muestras
    inicio_muestra = int(t1 * fs)
    fin_muestra = int(t2 * fs)
    
    metadata = {
        'ID_Sag': nombre_sag,
        'inicio_seg': t1,
        'fin_seg': t2,
        'inicio_muestra': inicio_muestra,
        'fin_muestra': fin_muestra,
        'alpha': alpha,
        'fase': fase
    }
    metadata_list.append(metadata)

# Guarda los metadatos en un DataFrame y luego en un archivo CSV
metadata_df = pd.DataFrame(metadata_list)
metadata_df.to_csv(r'Sag\sags_metadata.csv', index=False)

# Guarda el DataFrame de señales en un archivo CSV
sags_df.to_csv(r'Sag\sags.csv', index_label='Tiempo')

print(f"Se han generado {num_sags} sags.")
print("Señales guardadas en 'sags.csv'")
print("Metadatos (inicio/fin) guardados en 'sags_metadata.csv'")
print("\n--- Vista Previa de Metadatos ---")
print(metadata_df.head())

In [ ]:
import numpy as np
import pandas as pd
import os

def sag(amplitud, frecuencia, fase, tiempo, alpha, t1, t2):
    """Genera una señal de hundimiento de tensión."""
    titulo = "Hundimiento de Tensión"
    wt = 2 * np.pi * frecuencia * tiempo
    u1 = np.heaviside(tiempo - t1, 1)
    u2 = np.heaviside(tiempo - t2, 1)
    envolvente = 1 - (alpha * (u1 - u2))
    señal = amplitud * envolvente * np.sin(wt + fase)
    return señal, titulo

# --- Parámetros de Generación ---
num_sags_ruido = 3000
num_puntos = 7200
duracion_total = 1.0 # en segundos
fs = num_puntos / duracion_total # Frecuencia de muestreo

tiempo = np.linspace(0, duracion_total, num_puntos)
sags_ruido_df = pd.DataFrame()

# --- Nivel de Ruido ---
noise_level = 0.03 # Ruido del 3% de la amplitud máxima

# 1. Carga los metadatos existentes
metadata_path = r'Sag\sags_metadata.csv'
try:
    # Intenta cargar el archivo si ya existe
    existing_metadata_df = pd.read_csv(metadata_path)
    print(f"Metadatos existentes cargados desde '{metadata_path}'. Filas: {len(existing_metadata_df)}")
except FileNotFoundError:
    # Si no existe, crea un DataFrame vacío para empezar
    existing_metadata_df = pd.DataFrame()
    print(f"No se encontró '{metadata_path}'. Se creará un nuevo archivo de metadatos.")

# Inicializa una lista para los nuevos metadatos
new_metadata_list = []

# --- Bucle de Generación ---
for i in range(num_sags_ruido):
    # Genera parámetros aleatorios
    amplitud = 1.0
    frecuencia = 60.0
    fase = np.random.uniform(0, 2 * np.pi)
    alpha = np.random.uniform(0.2, 0.8)
    t1 = np.random.uniform(0.05, 0.1)
    t2 = t1 + np.random.uniform(0.05, 0.08)

    # Genera la señal limpia
    sag_signal, _ = sag(amplitud, frecuencia, fase, tiempo, alpha, t1, t2)

    # Genera y añade ruido Gaussiano
    ruido = np.random.normal(0, noise_level, sag_signal.shape)
    sag_con_ruido = sag_signal + ruido

    # Añade la señal con ruido al DataFrame
    nombre_sag = f'Sag_ruido_{i+1}'
    sags_ruido_df[nombre_sag] = sag_con_ruido

    # 2. Crea el registro de metadatos para la nueva señal
    inicio_muestra = int(t1 * fs)
    fin_muestra = int(t2 * fs)
    
    metadata = {
        'ID_Sag': nombre_sag,
        'inicio_seg': t1,
        'fin_seg': t2,
        'inicio_muestra': inicio_muestra,
        'fin_muestra': fin_muestra,
        'alpha': alpha,
        'fase': fase,
        'noise_level': noise_level # Añadimos el nivel de ruido
    }
    new_metadata_list.append(metadata)


# --- Guardado de Archivos ---

# Guarda el nuevo DataFrame de señales con ruido
sags_ruido_df.to_csv(r'Sag\sags_con_ruido.csv', index_label='Tiempo')
print(f"\nSe han generado {num_sags_ruido} sags con ruido y guardado en 'sags_con_ruido.csv'")

# 3. Combina los metadatos viejos y nuevos y guarda el archivo actualizado
new_metadata_df = pd.DataFrame(new_metadata_list)
combined_metadata_df = pd.concat([existing_metadata_df, new_metadata_df], ignore_index=True)

# Guarda el archivo de metadatos combinado, sobrescribiendo el anterior
combined_metadata_df.to_csv(metadata_path, index=False)
print(f"Metadatos actualizados y guardados en '{metadata_path}'. Total de filas: {len(combined_metadata_df)}")

# Muestra una vista previa de los últimos datos añadidos
print("\n--- Muestra de los últimos metadatos añadidos ---")
print(combined_metadata_df.tail())

## Sag_Armonico

In [ ]:
import numpy as np
import pandas as pd
import os
import json # Usaremos JSON para guardar los diccionarios de forma limpia

def generate_sag_with_random_harmonics(t, A, f, fase, t_inicio, t_fin, alpha, 
                                       background_harmonics={}, event_harmonics={}):
    """
    Genera un hundimiento de tensión con armónicos de fondo y/o durante el evento.
    """
    w = 2 * np.pi * f
    
    # 1. Generar la envolvente del hundimiento
    u1 = np.heaviside(t - t_inicio, 1)
    u2 = np.heaviside(t - t_fin, 1)
    envelope = 1 - ((1 - alpha) * (u1 - u2))
    
    # 2. Generar la señal fundamental con el hundimiento
    v_fundamental_sag = A * envelope * np.sin(w * t + fase)
    
    # 3. Generar y sumar los armónicos de fondo
    v_harm_background = np.zeros_like(t)
    for order, amp_fraction in background_harmonics.items():
        v_harm_background += (amp_fraction * A) * np.sin(order * w * t + fase)
        
    # 4. Generar los armónicos del evento
    v_harm_event = np.zeros_like(t)
    for order, amp_fraction in event_harmonics.items():
        v_harm_event += (amp_fraction * A) * np.sin(order * w * t + fase)
        
    # 5. Combinar todo
    v_final = v_fundamental_sag + v_harm_background
    
    # Sumar los armónicos del evento únicamente en la ventana del hundimiento
    idx_sag = (t >= t_inicio) & (t <= t_fin)
    v_final[idx_sag] += v_harm_event[idx_sag]
    
    return v_final

# --- CONFIGURACIÓN GENERAL ---
# Cantidades a generar
num_sags_harm_only = 1100
num_sags_harm_noise = 3000

# Parámetros de la señal
num_puntos = 7200
duracion_total = 1.0
fs = num_puntos / duracion_total
tiempo = np.linspace(0, duracion_total, num_puntos)
amplitud_pico = 1.0
frecuencia_hz = 60.0
posibles_harmonicos = [3, 5, 7, 9, 11, 13, 15, 17, 19, 21]
noise_level = 0.03

# Rutas de guardado
output_dir = 'Sag_Armonic'
os.makedirs(output_dir, exist_ok=True)
metadata_path = os.path.join(output_dir, 'metadata_consolidado.csv')

# --- CARGAR O INICIALIZAR METADATOS ---
try:
    existing_metadata_df = pd.read_csv(metadata_path)
    print(f"Metadatos existentes cargados desde '{metadata_path}'. Filas: {len(existing_metadata_df)}")
except FileNotFoundError:
    existing_metadata_df = pd.DataFrame()
    print(f"No se encontró archivo de metadatos. Se creará uno nuevo.")

# Lista para recolectar TODOS los nuevos metadatos de esta ejecución
new_metadata_list = []

# --- 1. GENERACIÓN DE SAGS CON ARMÓNICOS (SIN RUIDO) ---
print(f"\nIniciando la generación de {num_sags_harm_only} sags con armónicos (sin ruido)...")
sags_harm_df = pd.DataFrame()

for i in range(num_sags_harm_only):
    fase_rad = np.random.uniform(0, 2 * np.pi)
    alpha_sag = np.random.uniform(0.1, 0.9)
    t1 = np.random.uniform(0.1, 0.4)
    t2 = t1 + np.random.uniform(0.1, 0.5)
    
    bg_harmonics_dict = {}
    event_harmonics_dict = {}
    harmonic_scenario = np.random.choice(['fondo_solamente', 'evento_solamente', 'ambos'], p=[0.25, 0.25, 0.50])

    if harmonic_scenario in ['fondo_solamente', 'ambos']:
            for order in np.random.choice(posibles_harmonicos, np.random.randint(1, 3), replace=False):
                bg_harmonics_dict[int(order)] = np.random.uniform(0.01, 0.04)
                
    if harmonic_scenario in ['evento_solamente', 'ambos']:
            for order in np.random.choice(posibles_harmonicos, np.random.randint(1, 4), replace=False):
                event_harmonics_dict[int(order)] = np.random.uniform(0.02, 0.1)

    sag_signal = generate_sag_with_random_harmonics(
        tiempo, amplitud_pico, frecuencia_hz, fase_rad, t1, t2, alpha_sag,
        background_harmonics=bg_harmonics_dict,
        event_harmonics=event_harmonics_dict
    )
    
    nombre_sag = f'Sag_Harm_{i+1}'
    sags_harm_df[nombre_sag] = sag_signal

    new_metadata_list.append({
        'ID_Sag': nombre_sag,
        'tipo': 'armonicos_solo',
        'inicio_muestra': int(t1 * fs),
        'fin_muestra': int(t2 * fs),
        'alpha': alpha_sag,
        'noise_level': 0.0,
        'bg_harmonics': json.dumps(bg_harmonics_dict),
        'event_harmonics': json.dumps(event_harmonics_dict)
    })

# --- 2. GENERACIÓN DE SAGS CON ARMÓNICOS Y RUIDO ---
print(f"\nIniciando la generación de {num_sags_harm_noise} sags con armónicos y ruido...")
sags_harm_ruido_df = pd.DataFrame()

for i in range(num_sags_harm_noise):
    # Todo el siguiente bloque ahora está DENTRO del bucle for
    fase_rad = np.random.uniform(0, 2 * np.pi)
    alpha_sag = np.random.uniform(0.2, 0.8)
    t1 = np.random.uniform(0.05, 0.1)
    t2 = t1 + np.random.uniform(0.05, 0.08)
    
    bg_harmonics_dict = {}
    event_harmonics_dict = {}
    harmonic_scenario = np.random.choice(['fondo_solamente', 'evento_solamente', 'ambos'], p=[0.25, 0.25, 0.50])

    if harmonic_scenario in ['fondo_solamente', 'ambos']:
        for order in np.random.choice(posibles_harmonicos, np.random.randint(1, 3), replace=False):
            bg_harmonics_dict[int(order)] = np.random.uniform(0.01, 0.04)
            
    if harmonic_scenario in ['evento_solamente', 'ambos']:
        for order in np.random.choice(posibles_harmonicos, np.random.randint(1, 4), replace=False):
            event_harmonics_dict[int(order)] = np.random.uniform(0.02, 0.1)

    sag_con_harmonicos = generate_sag_with_random_harmonics(
        tiempo, amplitud_pico, frecuencia_hz, fase_rad, t1, t2, alpha_sag,
        background_harmonics=bg_harmonics_dict,
        event_harmonics=event_harmonics_dict
    )
    
    ruido = np.random.normal(0, noise_level, sag_con_harmonicos.shape)
    sag_final_con_ruido = sag_con_harmonicos + ruido
    
    nombre_sag = f'Sag_HarmRuido_{i+1}'
    sags_harm_ruido_df[nombre_sag] = sag_final_con_ruido

    new_metadata_list.append({
        'ID_Sag': nombre_sag,
        'tipo': 'armonicos_y_ruido',
        'inicio_muestra': int(t1 * fs),
        'fin_muestra': int(t2 * fs),
        'alpha': alpha_sag,
        'noise_level': noise_level,
        'bg_harmonics': json.dumps(bg_harmonics_dict),
        'event_harmonics': json.dumps(event_harmonics_dict)
    })

# --- 3. GUARDAR TODOS LOS ARCHIVOS ---

# Guardar archivos de señales
csv_harm_only_path = os.path.join(output_dir, 'sags_con_armonicos.csv')
sags_harm_df.to_csv(csv_harm_only_path, index_label='Tiempo')
print(f"\nSeñales con armónicos guardadas en '{csv_harm_only_path}'")

csv_harm_noise_path = os.path.join(output_dir, 'sags_con_armonicos_y_ruido.csv')
sags_harm_ruido_df.to_csv(csv_harm_noise_path, index_label='Tiempo')
print(f"Señales con armónicos y ruido guardadas en '{csv_harm_noise_path}'")

# Combinar metadatos y guardar
new_metadata_df = pd.DataFrame(new_metadata_list)
combined_metadata_df = pd.concat([existing_metadata_df, new_metadata_df], ignore_index=True)
combined_metadata_df.to_csv(metadata_path, index=False)

print(f"\n¡Proceso completado!")
print(f"Metadatos de TODAS las señales consolidados en '{metadata_path}'.")
print(f"Total de registros en metadatos: {len(combined_metadata_df)}")
print("\n--- Muestra de los últimos metadatos añadidos ---")
print(combined_metadata_df.tail())

## Sag_Multiple


In [ ]:
import numpy as np
import pandas as pd
import os

def sag_multiple(amplitud, frecuencia, fase, tiempo, alpha1, alpha2, t1, t2, t3):
    """
    Genera una señal con un hundimiento de tensión de múltiples etapas.
    """
    wt = 2 * np.pi * frecuencia * tiempo
    u1 = np.heaviside(tiempo - t1, 1)      # Inicio de la primera caída
    u2 = np.heaviside(tiempo - t2, 1)      # Inicio de la segunda caída (más profunda)
    u3 = np.heaviside(tiempo - t3, 1)      # Fin de todo el evento
    
    caida1 = alpha1 * (u1 - u3)
    caida2 = (alpha2 - alpha1) * (u2 - u3)
    
    envolvente = 1 - (caida1 + caida2)
    señal = amplitud * envolvente * np.sin(wt + fase)
    return señal

# --- CONFIGURACIÓN GENERAL ---
# Cantidades a generar
num_eventos_limpios = 1100
num_eventos_ruido = 3000

# Parámetros de la señal
num_puntos = 7200
duracion_total = 1.0
fs = num_puntos / duracion_total
tiempo = np.linspace(0, duracion_total, num_puntos)
amplitud = 1.0
frecuencia = 60.0
nivel_de_ruido = 0.03

# Rutas de guardado
output_dir = 'Sag_Multiple'
os.makedirs(output_dir, exist_ok=True)
metadata_path = os.path.join(output_dir, 'metadata_multiples.csv')

# --- CARGAR O INICIALIZAR METADATOS ---
try:
    existing_metadata_df = pd.read_csv(metadata_path)
    print(f"Metadatos existentes cargados desde '{metadata_path}'. Filas: {len(existing_metadata_df)}")
except FileNotFoundError:
    existing_metadata_df = pd.DataFrame()
    print(f"No se encontró archivo de metadatos. Se creará uno nuevo.")

# Lista para recolectar TODOS los nuevos metadatos de esta ejecución
new_metadata_list = []

# --- 1. GENERACIÓN DE SAGS MÚLTIPLES (SIN RUIDO) ---
print(f"\nGenerando {num_eventos_limpios} sags múltiples sin ruido...")
df_sags_multiples_limpios = pd.DataFrame()

for i in range(num_eventos_limpios):
    fase = np.random.uniform(0, 2 * np.pi)
    alpha1 = np.random.uniform(0.1, 0.5)
    alpha2 = np.random.uniform(alpha1 + 0.1, 0.9)
    
    t1 = np.random.uniform(0.1, 0.4)
    t2 = t1 + np.random.uniform(0.05, 0.2)
    t3 = t2 + np.random.uniform(0.1, 0.3)
    if t3 >= 0.95: t3 = 0.95

    sag_signal = sag_multiple(amplitud, frecuencia, fase, tiempo, alpha1, alpha2, t1, t2, t3)
    
    nombre_sag = f'Sag_Multi_{i+1}'
    df_sags_multiples_limpios[nombre_sag] = sag_signal

    new_metadata_list.append({
        'ID_Sag': nombre_sag,
        'tipo': 'multiple_limpio',
        'inicio_muestra': int(t1 * fs), # El evento completo inicia en t1
        'fin_muestra': int(t3 * fs),   # El evento completo termina en t3
        'alpha1': alpha1,
        'alpha2': alpha2,
        't1_muestra': int(t1 * fs),
        't2_muestra': int(t2 * fs),
        't3_muestra': int(t3 * fs),
        'noise_level': 0.0
    })

# --- 2. GENERACIÓN DE SAGS MÚLTIPLES (CON RUIDO) ---
print(f"\nGenerando {num_eventos_ruido} sags múltiples con ruido...")
df_sags_multiples_ruido = pd.DataFrame()

for i in range(num_eventos_ruido):
    fase = np.random.uniform(0, 2 * np.pi)
    alpha1 = np.random.uniform(0.1, 0.5)
    alpha2 = np.random.uniform(alpha1 + 0.1, 0.9)
    
    t1 = np.random.uniform(0.1, 0.4)
    t2 = t1 + np.random.uniform(0.05, 0.2)
    t3 = t2 + np.random.uniform(0.1, 0.3)
    if t3 >= 0.95: t3 = 0.95

    sag_signal = sag_multiple(amplitud, frecuencia, fase, tiempo, alpha1, alpha2, t1, t2, t3)
    ruido = np.random.normal(0, nivel_de_ruido, num_puntos)
    sag_con_ruido = sag_signal + ruido
    
    nombre_sag = f'Sag_MultiRuido_{i+1}'
    df_sags_multiples_ruido[nombre_sag] = sag_con_ruido

    new_metadata_list.append({
        'ID_Sag': nombre_sag,
        'tipo': 'multiple_con_ruido',
        'inicio_muestra': int(t1 * fs),
        'fin_muestra': int(t3 * fs),
        'alpha1': alpha1,
        'alpha2': alpha2,
        't1_muestra': int(t1 * fs),
        't2_muestra': int(t2 * fs),
        't3_muestra': int(t3 * fs),
        'noise_level': nivel_de_ruido
    })

# --- 3. GUARDAR TODOS LOS ARCHIVOS ---

# Guardar archivos de señales
path_limpios = os.path.join(output_dir, 'sags_multiples.csv')
df_sags_multiples_limpios.to_csv(path_limpios, index_label='Tiempo')
print(f"\nSeñales limpias guardadas en '{path_limpios}'")

path_ruido = os.path.join(output_dir, 'sags_multiples_con_ruido.csv')
df_sags_multiples_ruido.to_csv(path_ruido, index_label='Tiempo')
print(f"Señales con ruido guardadas en '{path_ruido}'")

# Combinar metadatos y guardar
new_metadata_df = pd.DataFrame(new_metadata_list)
combined_metadata_df = pd.concat([existing_metadata_df, new_metadata_df], ignore_index=True)
combined_metadata_df.to_csv(metadata_path, index=False)

print(f"\n¡Proceso completado!")
print(f"Metadatos de TODAS las señales consolidados en '{metadata_path}'.")
print(f"Total de registros en metadatos: {len(combined_metadata_df)}")
print("\n--- Muestra de los últimos metadatos añadidos ---")
print(combined_metadata_df.tail())

## Sag_Notch


In [ ]:
import numpy as np
import pandas as pd
import os
import json

# --- PARTE 1: FUNCIONES DE GENERACIÓN ---

def generate_sag_fault(t, A, f, fase, t_inicio, t_fin, alpha, delta_phi):
    """Función base que genera un hundimiento rectangular."""
    w = 2 * np.pi * f
    m = np.ones_like(t)
    delta_phi_t = np.zeros_like(t)
    idx_sag = (t >= t_inicio) & (t <= t_fin)
    m[idx_sag] = alpha
    idx_phase_jump = t >= t_inicio
    delta_phi_t[idx_phase_jump] = delta_phi
    v_sag = m * A * np.sin(w * t + fase + delta_phi_t)
    return v_sag

def generate_sag_overload(t, A, f, fase, t_inicio, t_fin, alpha):
    """Genera un hundimiento rectangular sin salto de fase."""
    return generate_sag_fault(t, A, f, fase, t_inicio, t_fin, alpha, delta_phi=0)

def generate_sag_with_notches(t, A, f, fase, t_inicio, t_fin, alpha, notch_params):
    """Genera un hundimiento rectangular con muescas (notches) periódicas."""
    w = 2 * np.pi * f
    T = 1/f
    v_base = generate_sag_overload(t, A, f, fase, t_inicio, t_fin, alpha)
    v_notches = np.zeros_like(t)
    
    depth = notch_params['depth'] * A
    width = notch_params['width'] * T
    ppr = notch_params['pulses_per_cycle']
    
    num_cycles = (t_fin - t_inicio) / T
    total_pulses = int(np.floor(num_cycles * ppr))
    if total_pulses > 0:
        notch_instants = np.linspace(t_inicio, t_fin, total_pulses, endpoint=False)
        for t_k in notch_instants:
            v_notches += depth * np.exp(-((t - t_k)**2) / (2 * (width**2)))
            
    return v_base - v_notches

# --- PARTE 2: SCRIPT PRINCIPAL DE GENERACIÓN ---

# --- CONFIGURACIÓN GENERAL ---
num_sags_limpios = 1100
num_sags_ruido =3000

amplitud_pico = 1.0
frecuencia_hz = 60.0
num_puntos = 7200
duracion_total = 1.0
fs = num_puntos / duracion_total
tiempo = np.linspace(0, duracion_total, num_puntos)
noise_level = 0.02

# Rutas de guardado
output_dir = 'Sag_Notch'
os.makedirs(output_dir, exist_ok=True)
metadata_path = os.path.join(output_dir, 'metadata_notches.csv')

# --- CARGAR O INICIALIZAR METADATOS ---
try:
    existing_metadata_df = pd.read_csv(metadata_path)
    print(f"Metadatos existentes cargados desde '{metadata_path}'. Filas: {len(existing_metadata_df)}")
except FileNotFoundError:
    existing_metadata_df = pd.DataFrame()
    print(f"No se encontró archivo de metadatos. Se creará uno nuevo.")

new_metadata_list = []

# --- TAREA 1: Generar hundimientos con muescas (SIN RUIDO) ---
print(f"\n--- Iniciando Tarea 1: Generación de {num_sags_limpios} hundimientos con muescas (sin ruido) ---")
sags_notches_df = pd.DataFrame()

for i in range(num_sags_limpios):
    fase_rad = np.random.uniform(0, 2 * np.pi)
    alpha_sag = np.random.uniform(0.8, 0.95)
    t1 = np.random.uniform(0.1, 0.4)
    t2 = t1 + np.random.uniform(0.2, 0.5)
    
    notch_params_dict = {
        'depth': np.random.uniform(0.05, 0.25),
        'width': np.random.uniform(0.005, 0.02),
        'pulses_per_cycle': int(np.random.choice([6, 12])) # Convertir a int para JSON
    }

    sag_con_notches = generate_sag_with_notches(
        tiempo, amplitud_pico, frecuencia_hz, fase_rad, t1, t2, alpha_sag,
        notch_params=notch_params_dict
    )
    
    nombre_sag = f'Sag_Notch_{i+1}'
    sags_notches_df[nombre_sag] = sag_con_notches

    new_metadata_list.append({
        'ID_Sag': nombre_sag,
        'tipo': 'muescas_limpio',
        'inicio_muestra': int(t1 * fs),
        'fin_muestra': int(t2 * fs),
        'alpha': alpha_sag,
        'noise_level': 0.0,
        'notch_params': json.dumps(notch_params_dict)
    })

# --- TAREA 2: Generar hundimientos con muescas y RUIDO ---
print(f"\n--- Iniciando Tarea 2: Generación de {num_sags_ruido} hundimientos con muescas y ruido ---")
sags_ruido_df = pd.DataFrame()

for i in range(num_sags_ruido):
    fase_rad = np.random.uniform(0, 2 * np.pi)
    alpha_sag = np.random.uniform(0.8, 0.95)
    t1 = np.random.uniform(0.1, 0.4)
    t2 = t1 + np.random.uniform(0.2, 0.5)
    
    notch_params_dict = {
        'depth': np.random.uniform(0.05, 0.25),
        'width': np.random.uniform(0.005, 0.02),
        'pulses_per_cycle': int(np.random.choice([6, 12])) # Convertir a int para JSON
    }

    sag_con_notches = generate_sag_with_notches(
        tiempo, amplitud_pico, frecuencia_hz, fase_rad, t1, t2, alpha_sag,
        notch_params=notch_params_dict
    )
    
    ruido = np.random.normal(0, noise_level, sag_con_notches.shape)
    sag_final_con_ruido = sag_con_notches + ruido
    
    nombre_sag = f'Sag_NotchRuido_{i+1}'
    sags_ruido_df[nombre_sag] = sag_final_con_ruido

    new_metadata_list.append({
        'ID_Sag': nombre_sag,
        'tipo': 'muescas_con_ruido',
        'inicio_muestra': int(t1 * fs),
        'fin_muestra': int(t2 * fs),
        'alpha': alpha_sag,
        'noise_level': noise_level,
        'notch_params': json.dumps(notch_params_dict)
    })

# --- PARTE 3: GUARDAR TODOS LOS ARCHIVOS ---

# Guardar archivos de señales
csv_path_1 = os.path.join(output_dir, 'sags_con_muescas.csv')
sags_notches_df.to_csv(csv_path_1, index_label='Tiempo')
print(f"\nSeñales limpias guardadas en '{csv_path_1}'")

csv_path_2 = os.path.join(output_dir, 'sags_con_muescas_y_ruido.csv')
sags_ruido_df.to_csv(csv_path_2, index_label='Tiempo')
print(f"Señales con ruido guardadas en '{csv_path_2}'")

# Combinar y guardar metadatos
new_metadata_df = pd.DataFrame(new_metadata_list)
combined_metadata_df = pd.concat([existing_metadata_df, new_metadata_df], ignore_index=True)
combined_metadata_df.to_csv(metadata_path, index=False)

print(f"\n¡Proceso completado!")
print(f"Metadatos de TODAS las señales consolidados en '{metadata_path}'.")
print(f"Total de registros en metadatos: {len(combined_metadata_df)}")
print("\n--- Muestra de los últimos metadatos añadidos ---")
print(combined_metadata_df.tail())

## Sag_Transitorios

In [14]:
import numpy as np
import pandas as pd
import os
import json

# --- PARTE 1: FUNCIONES DE GENERACIÓN ---

def generate_sag_fault(t, A, f, fase, t_inicio, t_fin, alpha, delta_phi):
    w = 2 * np.pi * f
    m = np.ones_like(t)
    delta_phi_t = np.zeros_like(t)
    idx_sag = (t >= t_inicio) & (t <= t_fin)
    m[idx_sag] = alpha
    idx_phase_jump = t >= t_inicio
    delta_phi_t[idx_phase_jump] = delta_phi
    v_sag = m * A * np.sin(w * t + fase + delta_phi_t)
    return v_sag

def generate_sag_overload(t, A, f, fase, t_inicio, t_fin, alpha):
    return generate_sag_fault(t, A, f, fase, t_inicio, t_fin, alpha, delta_phi=0)

def generate_sag_with_transient(t, A, f, fase, t_inicio, t_fin, alpha, transient_params):
    w = 2 * np.pi * f
    v_base = generate_sag_overload(t, A, f, fase, t_inicio, t_fin, alpha)
    v_transient = np.zeros_like(t)
    idx_transient = t >= t_inicio
    t_transient = t[idx_transient]
    A_t = transient_params['A_t_frac'] * A
    f_t = transient_params['f_t']
    w_t = 2 * np.pi * f_t
    tau_t = transient_params['tau_t']
    decay = np.exp(-(t_transient - t_inicio) / tau_t)
    oscillation = np.sin(w_t * (t_transient - t_inicio))
    v_transient[idx_transient] = A_t * decay * oscillation
    return v_base + v_transient

# --- PARTE 2: SCRIPT PRINCIPAL DE GENERACIÓN ---

# --- CONFIGURACIÓN GENERAL ---
num_sags_limpios = 1100
num_sags_ruido = 3000

amplitud_pico = 1.0
frecuencia_hz = 60.0
T_periodo = 1/frecuencia_hz
num_puntos = 7200
duracion_total = 1.0
fs = num_puntos / duracion_total
tiempo = np.linspace(0, duracion_total, num_puntos)
noise_level = 0.03

# Rutas de guardado
output_dir = 'Sag_Transitorios'
os.makedirs(output_dir, exist_ok=True)
metadata_path = os.path.join(output_dir, 'metadata_transitorios.csv')

# --- CARGAR O INICIALIZAR METADATOS ---
try:
    existing_metadata_df = pd.read_csv(metadata_path)
    print(f"Metadatos existentes cargados desde '{metadata_path}'. Filas: {len(existing_metadata_df)}")
except FileNotFoundError:
    existing_metadata_df = pd.DataFrame()
    print(f"No se encontró archivo de metadatos. Se creará uno nuevo.")

new_metadata_list = []

# --- TAREA 1: Generar hundimientos con transitorios (SIN RUIDO) ---
print(f"\n--- Iniciando Tarea 1: Generación de {num_sags_limpios} hundimientos con transitorios (sin ruido) ---")
sags_transient_df = pd.DataFrame() 

for i in range(num_sags_limpios):
    fase_rad = np.random.uniform(0, 2 * np.pi)
    alpha_sag = np.random.uniform(0.7, 0.9)
    t1 = np.random.uniform(0.1, 0.4)
    t2 = t1 + np.random.uniform(0.1, 0.5)
    
    transient_params_dict = {
        'A_t_frac': np.random.uniform(0.1, 0.4),
        'f_t': np.random.uniform(300, 1200),
        'tau_t': np.random.uniform(1.5, 5) * T_periodo
    }
    
    sag_con_transitorio = generate_sag_with_transient(
        tiempo, amplitud_pico, frecuencia_hz, fase_rad, t1, t2, alpha_sag,
        transient_params=transient_params_dict
    )
    
    nombre_sag = f'Sag_Trans_{i+1}'
    sags_transient_df[nombre_sag] = sag_con_transitorio

    new_metadata_list.append({
        'ID_Sag': nombre_sag,
        'tipo': 'transitorio_limpio',
        'inicio_muestra': int(t1 * fs),
        'fin_muestra': int(t2 * fs),
        'alpha': alpha_sag,
        'noise_level': 0.0,
        'transient_params': json.dumps(transient_params_dict)
    })

# --- TAREA 2: Generar hundimientos con transitorios y RUIDO ---
print(f"\n--- Iniciando Tarea 2: Generación de {num_sags_ruido} hundimientos con transitorios y ruido ---")
sags_ruido_df = pd.DataFrame()

for i in range(num_sags_ruido):
    fase_rad = np.random.uniform(0, 2 * np.pi)
    alpha_sag = np.random.uniform(0.7, 0.9)
    t1 = np.random.uniform(0.1, 0.4)
    t2 = t1 + np.random.uniform(0.1, 0.5)
    
    transient_params_dict = {
        'A_t_frac': np.random.uniform(0.1, 0.4),
        'f_t': np.random.uniform(300, 1200),
        'tau_t': np.random.uniform(1.5, 5) * T_periodo
    }
    
    sag_con_transitorio = generate_sag_with_transient(
        tiempo, amplitud_pico, frecuencia_hz, fase_rad, t1, t2, alpha_sag,
        transient_params=transient_params_dict
    )
    
    ruido = np.random.normal(0, noise_level, sag_con_transitorio.shape)
    sag_final_con_ruido = sag_con_transitorio + ruido
    
    nombre_sag = f'Sag_TransRuido_{i+1}'
    sags_ruido_df[nombre_sag] = sag_final_con_ruido

    new_metadata_list.append({
        'ID_Sag': nombre_sag,
        'tipo': 'transitorio_con_ruido',
        'inicio_muestra': int(t1 * fs),
        'fin_muestra': int(t2 * fs),
        'alpha': alpha_sag,
        'noise_level': noise_level,
        'transient_params': json.dumps(transient_params_dict)
    })

# --- PARTE 3: GUARDAR TODOS LOS ARCHIVOS ---

# Guardar archivos de señales
csv_path_1 = os.path.join(output_dir, 'sags_con_transitorios.csv')
sags_transient_df.to_csv(csv_path_1, index_label='Tiempo')
print(f"\nSeñales limpias guardadas en '{csv_path_1}'")

csv_path_2 = os.path.join(output_dir, 'sags_con_transitorios_y_ruido.csv')
sags_ruido_df.to_csv(csv_path_2, index_label='Tiempo')
print(f"Señales con ruido guardadas en '{csv_path_2}'")

# Combinar y guardar metadatos
new_metadata_df = pd.DataFrame(new_metadata_list)
combined_metadata_df = pd.concat([existing_metadata_df, new_metadata_df], ignore_index=True)
combined_metadata_df.to_csv(metadata_path, index=False)

print(f"\n¡Proceso completado!")
print(f"Metadatos de TODAS las señales consolidados en '{metadata_path}'.")
print(f"Total de registros en metadatos: {len(combined_metadata_df)}")
print("\n--- Muestra de los últimos metadatos añadidos ---")
print(combined_metadata_df.tail())

No se encontró archivo de metadatos. Se creará uno nuevo.

--- Iniciando Tarea 1: Generación de 1100 hundimientos con transitorios (sin ruido) ---


C:\Users\Juan_Saa\AppData\Local\Temp\ipykernel_28212\435932729.py:89: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  sags_transient_df[nombre_sag] = sag_con_transitorio
C:\Users\Juan_Saa\AppData\Local\Temp\ipykernel_28212\435932729.py:89: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  sags_transient_df[nombre_sag] = sag_con_transitorio
C:\Users\Juan_Saa\AppData\Local\Temp\ipykernel_28212\435932729.py:89: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has 


--- Iniciando Tarea 2: Generación de 3000 hundimientos con transitorios y ruido ---


C:\Users\Juan_Saa\AppData\Local\Temp\ipykernel_28212\435932729.py:126: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  sags_ruido_df[nombre_sag] = sag_final_con_ruido
C:\Users\Juan_Saa\AppData\Local\Temp\ipykernel_28212\435932729.py:126: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  sags_ruido_df[nombre_sag] = sag_final_con_ruido
C:\Users\Juan_Saa\AppData\Local\Temp\ipykernel_28212\435932729.py:126: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor 


Señales limpias guardadas en 'Sag_Transitorios\sags_con_transitorios.csv'
Señales con ruido guardadas en 'Sag_Transitorios\sags_con_transitorios_y_ruido.csv'

¡Proceso completado!
Metadatos de TODAS las señales consolidados en 'Sag_Transitorios\metadata_transitorios.csv'.
Total de registros en metadatos: 4100

--- Muestra de los últimos metadatos añadidos ---
                   ID_Sag                   tipo  inicio_muestra  fin_muestra  \
4095  Sag_TransRuido_2996  transitorio_con_ruido            1531         2367   
4096  Sag_TransRuido_2997  transitorio_con_ruido            1571         4780   
4097  Sag_TransRuido_2998  transitorio_con_ruido            1441         2950   
4098  Sag_TransRuido_2999  transitorio_con_ruido            1955         3234   
4099  Sag_TransRuido_3000  transitorio_con_ruido            1001         3116   

         alpha  noise_level                                   transient_params  
4095  0.725474         0.03  {"A_t_frac": 0.23418533120714538, "f_t":